In [214]:
import pandas as pd
from IPython.display import Markdown, display
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sympy import false
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from sklearn.model_selection import LeaveOneOut



In [215]:
def tableHeader():
    doc = "|Algorithm| label | accuracy | precision | recall | f1 |\n"
    doc += "| --- | --- | --- | --- |--- | --- |\n"
    return doc

def tableScore(labels, doc, prefix=None, y_true=None, y_prediction=None):
    rows = ""
    accuracy = accuracy_score(y_true, y_prediction)
    precisions = precision_score(y_true, y_prediction, labels=labels, average=None)
    recalls = recall_score(y_true, y_prediction, labels=labels, average=None)
    f1s = f1_score(y_true, y_prediction, labels=labels, average=None)
    for label, precision, recall, f1 in zip(labels, precisions, recalls, f1s):
        rows += f"| {prefix} | {label} | {accuracy:.4g} | {precision:.4g} | {recall:.4g} | {f1:.4g} |\n"
        prefix = " "
    return doc + rows

cross_validation_scores = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1_score': make_scorer(f1_score, average='weighted')
}

def tableCVScore(labels, doc, prefix=None, scores=None):
    rows = ""
    accuracy = scores['test_accuracy']
    precision = scores['test_precision']
    recall = scores['test_recall']
    f1 = scores['test_f1_score']
    rows += f"| {prefix} | * | {accuracy:.4g} | {precision:.4g} | {recall:.4g} | {f1:.4g} |\n"
    return doc + rows


def tableFooter(doc):
    display(Markdown(doc))

enable_debug = false

def debug(string):
    if (enable_debug):
        print(string)

# https://xkcd.com/221/ - 4 is overused
random_seed = 221

# Exploring Classification Algorithms

## Data Loading

In [216]:
df_csv = pd.read_csv('data/breast-cancer.csv')

target_column = 'diagnosis'
id_column = 'id'

feature_names = df_csv.columns.tolist()
debug(feature_names)
df_csv = df_csv.drop(id_column, axis=1)

Y_df = df_csv[[target_column]]
X_df = df_csv.drop([target_column], axis=1)
# unique converts to an NDArray, so the sort needs to happen first.
target_columns = Y_df.columns.tolist()
debug(f"target column: {target_columns}")
feature_names = X_df.columns.tolist()
debug(f"feature names: {feature_names}")
labels = Y_df[target_column].sort_values().unique()
debug(f"labels: {labels}")

# Split the data into Train and Test
X_train, X_test, Y_train, Y_test = train_test_split(
    X_df, Y_df, test_size= 0.3, random_state=random_seed)

## Missing Data Processing and Data Normalization

1. normalise using standardisation
2. normalise using min-max scaling
3. Create KNN model with both
4. Create DT model with both

In [217]:
mean_imputer = SimpleImputer(missing_values=0, strategy='mean')
mean_imputer.fit(X_train)
X_train_mean = mean_imputer.transform(X_train)
X_test_mean = mean_imputer.transform(X_test)
X_df_mean = mean_imputer.transform(X_df)

median_imputer = SimpleImputer(missing_values=0, strategy='median')
median_imputer.fit(X_train)
X_train_median = median_imputer.transform(X_train)
X_test_median = median_imputer.transform(X_test)
X_df_median = median_imputer.transform(X_df)

In [223]:
def crossValidateUsingTree(random_state=None, doc=None, prefix=None, X=None, Y=None):
    tree = DecisionTreeClassifier(criterion="entropy", random_state=random_state)
    cv_scores = cross_validate(tree, X, Y, cv=5, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores).mean()
    cv_scores_dev_df = pd.DataFrame(cv_scores).std()
    doc = tableCVScore(labels, doc, f"{prefix}, mean", cv_scores_df)
    # doc = tableCVScore(labels, doc, f"{prefix}, stddev", cv_scores_dev_df)
    return doc

def crossValidateUsingKnn(doc=None, X=None, Y=None, prefix=None):
    knn = KNeighborsClassifier()
    knn_cv_scores = cross_validate(knn, X, Y[target_column], cv=5, scoring=cross_validation_scores)
    knn_cv_scores_df = pd.DataFrame(knn_cv_scores).mean()
    knn_cv_scores_dev_df = pd.DataFrame(knn_cv_scores).std()
    doc = tableCVScore(labels, doc, f"{prefix}, mean", knn_cv_scores_df)
    # doc = tableCVScore(labels, doc, f"{prefix}, stddev", knn_cv_scores_dev_df)
    return doc


## DecisionTree evaluation of imputation methods

In [224]:
doc = tableHeader()

doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="none", X=X_train, Y=Y_train)
doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="mean", X=X_train_mean, Y=Y_train)
doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="median", X=X_train_median, Y=Y_train)

doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="full mean", X=X_df_mean, Y=Y_df)
doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="full median", X=X_df_median, Y=Y_df)

doc = tableFooter(doc)


|Algorithm| label | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- |
| none, mean | * | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
| mean, mean | * | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
| median, mean | * | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
| full mean, mean | * | 0.9244 | 0.9276 | 0.9244 | 0.9242 |
| full median, mean | * | 0.9262 | 0.929 | 0.9262 | 0.9259 |


Once the DecisionTree had a fixed seed, both imputations methods were the same, and performed "the same" as no imputation. Using "median" going forward. Interestingly, they both performed worse than leaving the values unset.

Decision trees are surprisingly tolerant of missing data.

Adding in median and mean cross validation using the full dataset, we see that median works slightly better than mean.

In [225]:
X_train = X_train_median
X_test = X_test_median
baseline_cv_scores_df = median_imputation_cv_scores_df
baseline_cv_scores_dev_df = median_imputation_cv_scores_dev_df

## Standardization

In [226]:
standard_scaler = StandardScaler()
standard_scaler.fit(X_train)
X_train_standardized = standard_scaler.transform(X_train, copy=True)
X_test_standardized = standard_scaler.transform(X_test, copy=True)
X_df_standardized = standard_scaler.transform(X_df_median, copy=True)

min_max_scaler = MinMaxScaler(copy=True) # don't clip. Why is MinMaxScaler different to StandardizedScaler?!?
min_max_scaler.fit(X_train)
X_train_min_max = min_max_scaler.transform(X_train)
X_test_min_max = min_max_scaler.transform(X_test)
X_df_min_max = min_max_scaler.transform(X_df_median)


In [230]:
doc = tableHeader()

doc = tableCVScore(labels, doc, "dt baseline, mean", baseline_cv_scores_df)
doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="dt standardized", X=X_train_standardized, Y=Y_train)
doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="dt min_max", X=X_train_min_max, Y=Y_train)
doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="dt full standardized", X=X_df_standardized, Y=Y_df)
doc = crossValidateUsingTree(random_state=random_seed, doc=doc, prefix="dt full min_max", X=X_df_min_max, Y=Y_df)

doc = crossValidateUsingKnn(doc=doc, prefix="knn baseline", X=X_train, Y=Y_train)
doc = crossValidateUsingKnn(doc=doc, prefix="knn standardized", X=X_train_standardized, Y=Y_train)
doc = crossValidateUsingKnn(doc=doc, prefix="knn min-max", X=X_train_min_max, Y=Y_train)
doc = crossValidateUsingKnn(doc=doc, prefix="knn full standardized", X=X_df_standardized, Y=Y_df)
doc = crossValidateUsingKnn(doc=doc, prefix="knn full min-max", X=X_df_min_max, Y=Y_df)

doc = tableFooter(doc)


|Algorithm| label | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- |
| dt baseline, mean | * | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
| dt standardized, mean | * | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
| dt min_max, mean | * | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
| dt full standardized, mean | * | 0.9262 | 0.929 | 0.9262 | 0.9259 |
| dt full min_max, mean | * | 0.9262 | 0.929 | 0.9262 | 0.9259 |
| knn baseline, mean | * | 0.9598 | 0.9614 | 0.9598 | 0.9593 |
| knn standardized, mean | * | 0.9598 | 0.9614 | 0.9598 | 0.9593 |
| knn min-max, mean | * | 0.9624 | 0.9637 | 0.9624 | 0.9619 |
| knn full standardized, mean | * | 0.9666 | 0.9675 | 0.9666 | 0.9664 |
| knn full min-max, mean | * | 0.9701 | 0.9706 | 0.9701 | 0.97 |


There is no benefit to standardization over baseline in decision trees for either normalisation method. This makes sense because the tree is splitting at a threshold, so the number of thresholds doesn't change with scale changes.

KNN shows an improvement using min-max, and this continues through to the full dataset.

In [228]:
X_train = X_train_min_max
X_test = X_test_min_max

1. KNN with hyperparameters
2. DecisionTree with hyperparameters
3. AdaBoost with hyperparameters - defaults to a DecisionTree.
4. Random Forest with hyperparameters


1. Pearson Correlation... threshold = 0.6
2. report the correlation matrix
3. manually retain features.
3. find features with threshold > 0.6
3. train a dt on the retained features..

1. feature extraction using PCA
2. evaluate using DT
3. max_depth = 2 and 8
4. don't forget to normalize before doing PCA - uses standardization, not min-max.
5. demonstrate transforming the test data.